<a href="https://colab.research.google.com/github/SandeshxS/Poultry-Disease-Classification/blob/main/EfficientNetV2_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

In [ ]:
BASE = "/content/drive/MyDrive/review2/chicken_disease"
train_dir = f"{BASE}/train"
val_dir = f"{BASE}/validation"
test_dir = f"{BASE}/test"

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
train_dir,
image_size=(IMG_SIZE, IMG_SIZE),
batch_size=BATCH_SIZE,
label_mode='categorical',
shuffle=True,
seed=SEED
)
# IMPORTANT: extract class names BEFORE prefetch
class_names = train_ds_raw.class_names
class_names

In [ ]:
train_ds = train_ds_raw.prefetch(AUTOTUNE)

In [ ]:
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
val_dir,
image_size=(IMG_SIZE, IMG_SIZE),
batch_size=BATCH_SIZE,
label_mode='categorical',
shuffle=False
).prefetch(AUTOTUNE)

In [ ]:
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
test_dir,
image_size=(IMG_SIZE, IMG_SIZE),
batch_size=BATCH_SIZE,
label_mode='categorical',
shuffle=False
).prefetch(AUTOTUNE)

In [ ]:
class_counts = {
i: len(os.listdir(os.path.join(train_dir, cls)))
for i, cls in enumerate(class_names)
}
total = sum(class_counts.values())
class_weights = {cls: total/class_counts[cls] for cls in class_counts}
class_weights

In [ ]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip("horizontal"),
tf.keras.layers.RandomRotation(0.1),
tf.keras.layers.RandomZoom(0.1),
])

In [ ]:
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(
include_top=False,
weights='imagenet',
input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
optimizer=tf.keras.optimizers.Adam(1e-3),
loss="categorical_crossentropy",
metrics=["accuracy"]
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
monitor='val_accuracy', patience=7, restore_best_weights=True
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.h5",
     save_best_only=True,
     monitor="val_accuracy"
)

In [ ]:
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=50,
class_weight=class_weights,
callbacks=[early_stop, checkpoint]
)

In [ ]:
base_model.trainable = True
model.compile(
optimizer=tf.keras.optimizers.Adam(1e-4),
loss="categorical_crossentropy",
metrics=["accuracy"]
)
history_ft = model.fit(
train_ds,
validation_data=val_ds,
epochs=20,
class_weight=class_weights,
callbacks=[early_stop]
)

In [ ]:
def plot_curves(history, title=""):
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(acc); plt.plot(val_acc)
plt.legend(["Train", "Val"])
plt.title("Accuracy " + title)
plt.subplot(1,2,2)
plt.plot(loss); plt.plot(val_loss)
plt.legend(["Train", "Val"])
plt.title("Loss " + title)
plt.show()
plot_curves(history, "(Stage 1)")
plot_curves(history_ft, "(Stage 2)")

In [ ]:
y_true = []
y_pred = []
for x, y in test_ds:
preds = model.predict(x)
y_true.extend(np.argmax(y, axis=1))
y_pred.extend(np.argmax(preds, axis=1))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, cmap="Blues",

xticklabels=class_names, yticklabels=class_names)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
y_prob = np.concatenate([model.predict(x) for x, _ in test_ds])
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))
plt.figure(figsize=(9,7))
for i, cls in enumerate(class_names):
fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
plt.plot(fpr, tpr, label=f"{cls} AUC={auc(fpr,tpr):.2f}")
plt.plot([0,1],[0,1],"k--")
plt.legend()
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.show()

In [ ]:
import os, shutil, tensorflow as tf
save_path = "/content/drive/MyDrive/efficientnet_model"
if os.path.exists(save_path):
shutil.rmtree(save_path)
os.makedirs(save_path, exist_ok=True)
model.save(f"{save_path}/model.h5")
model.export(f"{save_path}/saved_model")
print("Saved to:", save_path)